# `sys.path` 与跨目录导入模块

在 `token_tracker_explained.ipynb` 中有这两行代码：
```python
import sys
sys.path.insert(0, '../scripts')

from token_tracker import TokenTracker
```

本 notebook 解释：为什么需要这样做？`sys.path` 是什么？

## 1. 问题：为什么直接 import 找不到？

项目目录结构：
```
biojson/
├── scripts/
│   └── token_tracker.py      <-- 模块在这里
├── self-learn/
│   └── token_tracker_explained.ipynb  <-- notebook 在这里
```

当 notebook 执行 `from token_tracker import TokenTracker` 时，
Python 会在一系列目录中查找 `token_tracker.py`。

**但默认不会去 `scripts/` 目录找！** 所以会报 `ModuleNotFoundError`。

## 2. `sys.path` 是什么？

**`sys.path` 是一个列表**，包含了 Python 查找模块时会搜索的所有目录路径。

Python 执行 `import xxx` 时，会按 `sys.path` 中的顺序**逐个目录查找** `xxx.py`。

In [4]:
import sys

print("sys.path 内容（每行一个路径）：\n")
for i, p in enumerate(sys.path):
    print(f"  [{i}] {p}")

sys.path 内容（每行一个路径）：

  [0] /data/haotianwu/biojson/scripts
  [1] /data/hyj/miniconda/envs/biojson/lib/python311.zip
  [2] /data/hyj/miniconda/envs/biojson/lib/python3.11
  [3] /data/hyj/miniconda/envs/biojson/lib/python3.11/lib-dynload
  [4] 
  [5] /data/hyj/miniconda/envs/biojson/lib/python3.11/site-packages


## 3. 解决方案：把目标目录加入 sys.path

```python
sys.path.insert(0, '../scripts')
```

| 部分 | 含义 |
|------|------|
| `sys.path` | Python 的模块搜索路径列表 |
| `.insert(0, ...)` | 插入到列表**最前面**（优先搜索） |
| `'../scripts'` | 相对路径：从 notebook 所在目录往上一级，再进入 scripts/ |

加入后，Python 就能在 `scripts/` 目录下找到 `token_tracker.py` 了。

In [5]:
# 演示：加入路径前后的变化

target = "../scripts"

# 检查目标路径是否已在 sys.path 中
import os
abs_target = os.path.abspath(target)
print(f"目标绝对路径: {abs_target}")
print(f"已在 sys.path 中? {abs_target in sys.path}")

# 加入
if abs_target not in sys.path:
    sys.path.insert(0, abs_target)
    print("已添加到 sys.path")

# 现在可以 import 了
from token_tracker import TokenTracker
print(f"\n导入成功！TokenTracker = {TokenTracker}")

目标绝对路径: /data/haotianwu/biojson/scripts
已在 sys.path 中? True

导入成功！TokenTracker = <class 'token_tracker.TokenTracker'>


## 4. `insert(0, ...)` vs `append(...)`

| 方法 | 效果 | 优先级 |
|------|------|--------|
| `sys.path.insert(0, path)` | 加到最前面 | 最先搜索 |
| `sys.path.append(path)` | 加到最后面 | 最后搜索 |

一般用 `insert(0, ...)` 确保我们的模块优先被找到，避免和其他同名模块冲突。

In [3]:
# 对比 insert 和 append

demo_list = ['a', 'b', 'c']
print(f"原始: {demo_list}")

demo_list.insert(0, 'X')   # 插到最前面
print(f"insert(0, X): {demo_list}")

demo_list.append('Y')      # 加到最后面
print(f"append(Y):    {demo_list}")

原始: ['a', 'b', 'c']
insert(0, X): ['X', 'a', 'b', 'c']
append(Y):    ['X', 'a', 'b', 'c', 'Y']


## 5. 其他跨目录导入方式（了解即可）

| 方式 | 说明 | 适用场景 |
|------|------|----------|
| `sys.path.insert()` | 手动添加搜索路径 | notebook、快速脚本 |
| `pip install -e .` | 可编辑安装整个包 | 正式项目 |
| `PYTHONPATH` 环境变量 | 在 shell 中设置 | CI/CD、服务器 |
| 相对导入 `from . import` | 包内模块互相导入 | 包内部 |

对于学习用的 notebook，`sys.path.insert` 是最简单直接的方式。

## 6. 小结

| 要点 | 说明 |
|------|------|
| `sys.path` | Python 查找模块的目录列表 |
| 默认不含其他子目录 | 所以跨目录 import 会失败 |
| `sys.path.insert(0, path)` | 把目标目录加到搜索路径最前面 |
| `'../scripts'` | 相对路径，从当前目录向上一级再进入 scripts/ |
| 只需执行一次 | 之后整个 notebook 都能 import 该目录的模块 |